In [1]:
import sys
from pathlib import Path

import numpy as np

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

REPO_ROOT = Path(r"C:\Projects\signia-fsl-recognition").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_collection.dataset_builder import DatasetBuilder

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

COLLECTED_DIR = REPO_ROOT / "scripts" / "data" / "collected"
LABELS_CSV = REPO_ROOT / "csv" / "labels.csv"

OUTPUT_PATH = (
    REPO_ROOT
    / "data"
    / "test_dataset.pt"
)


# ------------------------------------------------------------
# Main verification
# ------------------------------------------------------------

def main():

    print("=" * 70)
    print("DATASET BUILDER VERIFICATION")
    print("=" * 70)

    print(f"Collected directory:")
    print(f"  {COLLECTED_DIR}")

    print(f"\nLabels CSV:")
    print(f"  {LABELS_CSV}")

    # --------------------------------------------------------
    # Build dataset
    # --------------------------------------------------------

    builder = DatasetBuilder(
        collected_dir=COLLECTED_DIR,
        labels_csv=LABELS_CSV,
    )

    print("\nBuilding dataset...")

    X, y = builder.build_dataset()

    # --------------------------------------------------------
    # Basic validation
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("DATASET RESULT")
    print("=" * 70)

    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")

    print(f"X dtype: {X.dtype}")
    print(f"y dtype: {y.dtype}")

    # Expected:
    # X = (N, 30, 126)
    # y = (N,)

    if X.ndim != 3:
        raise ValueError(
            f"X should have 3 dimensions, got {X.ndim}"
        )

    if y.ndim != 1:
        raise ValueError(
            f"y should have 1 dimension, got {y.ndim}"
        )

    if X.shape[0] != y.shape[0]:
        raise ValueError(
            f"X/y sample mismatch: "
            f"{X.shape[0]} != {y.shape[0]}"
        )

    if X.shape[1:] != (30, 126):
        raise ValueError(
            f"Unexpected sequence shape: {X.shape[1:]}. "
            f"Expected (30, 126)."
        )

    print("\nShape validation: PASSED")

    # --------------------------------------------------------
    # NaN / Inf check
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("DATA QUALITY")
    print("=" * 70)

    nan_count = np.isnan(X).sum()
    inf_count = np.isinf(X).sum()

    print(f"NaN values: {nan_count}")
    print(f"Inf values: {inf_count}")

    if nan_count > 0:
        print("WARNING: Dataset contains NaN values.")

    if inf_count > 0:
        print("WARNING: Dataset contains Inf values.")

    # --------------------------------------------------------
    # Value statistics
    # --------------------------------------------------------

    print("\nFeature statistics:")

    if len(X) > 0:
        print(f"Min:  {X.min():.6f}")
        print(f"Max:  {X.max():.6f}")
        print(f"Mean: {X.mean():.6f}")
        print(f"Std:  {X.std():.6f}")

    # --------------------------------------------------------
    # Class distribution
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("CLASS DISTRIBUTION")
    print("=" * 70)

    if len(y) == 0:
        print("WARNING: Dataset is empty.")
        return

    unique_labels, counts = np.unique(
        y,
        return_counts=True,
    )

    for label, count in zip(unique_labels, counts):
        print(f"Label {int(label):3d}: {count:4d} sequences")

    # --------------------------------------------------------
    # Dataset summary
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(f"Total sequences : {len(X)}")
    print(f"Total classes   : {len(unique_labels)}")
    print(f"Sequence shape  : {X.shape[1:]}")

    # --------------------------------------------------------
    # Save test dataset
    # --------------------------------------------------------

    print("\nSaving test dataset...")

    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Use torch so this has the same general structure
    # as your existing dataset bundle.
    import torch

    torch.save(
        {
            "X": torch.tensor(
                X,
                dtype=torch.float32,
            ),
            "y": torch.tensor(
                y,
                dtype=torch.long,
            ),
        },
        OUTPUT_PATH,
    )

    print(f"Saved: {OUTPUT_PATH}")

    # --------------------------------------------------------
    # Final verification
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("VERIFICATION PASSED")
    print("=" * 70)

    print(
        "The collected sequences can be successfully "
        "converted into an X/y dataset."
    )


if __name__ == "__main__":
    main()

DATASET BUILDER VERIFICATION
Collected directory:
  C:\Projects\signia-fsl-recognition\scripts\data\collected

Labels CSV:
  C:\Projects\signia-fsl-recognition\csv\labels.csv

Building dataset...

DATASET RESULT
X shape: (2, 30, 126)
y shape: (2,)
X dtype: float64
y dtype: int32

Shape validation: PASSED

DATA QUALITY
NaN values: 0
Inf values: 0

Feature statistics:
Min:  -1.000000
Max:  0.981998
Mean: -0.105180
Std:  0.252836

CLASS DISTRIBUTION
Label   0:    2 sequences

SUMMARY
Total sequences : 2
Total classes   : 1
Sequence shape  : (30, 126)

Saving test dataset...
Saved: C:\Projects\signia-fsl-recognition\data\test_dataset.pt

VERIFICATION PASSED
The collected sequences can be successfully converted into an X/y dataset.
